# Scenario Playground Step 14: Corrected All-Target Ridge Inference

Objective: demonstrate one callable baseline-versus-scenario workflow containing 12 independent target-specific Ridge pipelines over the frozen 58-feature schema.

All 12 targets return numeric Ridge baseline, scenario, and incremental predictions. Step 13 confidence evidence and warnings remain unchanged, and naive strategies remain comparison benchmarks rather than output replacements. This notebook does not read locked final-test targets, serialize models, expand XGBoost, implement an API, or begin Steps 15–17.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "scenario_modeling").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Run this notebook from inside the InsightPulse repository.")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scenario_modeling.src.scenario_inference import (
    ALL_RIDGE_TARGETS,
    EXAMPLE_SCENARIO,
    LOW_CONFIDENCE_PROMOTED_TARGETS,
    NEUTRAL_SCENARIO,
    ScenarioValidationError,
    WEAK_EVIDENCE_RIDGE_TARGETS,
    build_inference_artifacts,
    build_research_inference_engine,
    persist_inference_artifacts,
    predict_scenario,
)

engine = build_research_inference_engine()
artifacts = build_inference_artifacts(engine=engine)
output_paths = persist_inference_artifacts(artifacts)
summary = artifacts["summary"]
{
    "python_executable": sys.executable,
    "completion_gate": summary["completion_gate"],
    "latest_state": summary["latest_state_metadata"],
    "ridge_pipeline_count": summary["ridge_pipeline_count"],
    "ridge_targets": summary["ridge_targets"],
    "low_confidence_targets": summary["low_confidence_promoted_targets"],
    "weak_evidence_targets": summary["weak_evidence_ridge_targets"],
    "naive_substitutions": summary["naive_benchmark_substitution_count"],
    "locked_final_test": summary["locked_final_test"],
    "output_paths": output_paths,
}


## Exact 58-feature assembly

The row contains eight user-shock features, five internal release flags, and 45 current-state features. The PMI contract maps explicitly to the Kansas City Fed manufacturing proxy feature. The unsafe oil-volatility feature removed in Step 9 remains excluded.

In [ ]:
mapping = artifacts["feature_mapping"]
display(mapping.groupby("inference_role").size().rename("feature_count").to_frame())
display(mapping.loc[
    mapping["inference_role"].isin(["scenario_shock", "release_flag"]),
    ["feature_position", "feature_name", "inference_role", "contract_input", "unit", "baseline_policy"],
])
assert len(mapping) == 58
assert mapping["inference_role"].value_counts().to_dict() == {
    "current_state": 45,
    "scenario_shock": 8,
    "release_flag": 5,
}
assert "oil_volatility_20d_annualized_pct" not in set(mapping["feature_name"])


## Callable sample scenario and fixed context

The public entry point is `predict_scenario(scenario_shocks, ...)`. By default, a nonzero event shock activates its corresponding release flag once; that identical flag row is then used for both baseline and scenario. An explicit five-flag object can be supplied for a zero-surprise release or another research context.

In [ ]:
sample_scenario = dict(EXAMPLE_SCENARIO)
result = predict_scenario(sample_scenario, engine=engine)
display(pd.Series(result["scenario_shocks"], name="value").to_frame())
display(pd.Series(result["release_flags"], name="fixed_value").to_frame())
display(pd.Series({
    "sample_date": result["context"]["sample_date"],
    "state_cutoff_date": result["context"]["state_cutoff_date"],
    "state_source": result["context"]["state_source"],
    "state_feature_count": result["context"]["state_feature_count"],
}, name="context").to_frame())
display(pd.DataFrame(result["input_range_warnings"]))
assert result["release_flags"]["cpi_release_flag"] == 1.0
assert result["locked_final_test"]["access_count"] == 0


## Baseline/scenario isolation

The baseline has neutral values for all eight shock magnitudes. The scenario changes only the four nonzero shock columns in this sample. The 45 state features and all five release flags are byte-for-byte unchanged between the two rows.

In [ ]:
isolation = result["feature_assembly"]
display(pd.Series(isolation, name="value").to_frame())
assert isolation["feature_order_exact"]
assert isolation["only_shock_magnitudes_changed"]
assert isolation["state_identical"]
assert isolation["release_flags_identical"]
assert set(isolation["changed_features"]) == {
    "cpi_surprise_pct",
    "oil_price_change_pct",
    "dxy_change_pct",
    "vix_change_points",
}


## Complete 12-target output

Every target preserves numeric Ridge baseline, scenario, and incremental values separately. The confidence column keeps the unchanged Step 13 evidence visible: three targets passed the prior low-confidence gate and nine remain weak-evidence experiments that did not reliably outperform their best naive benchmark.

In [ ]:
output_rows = []
for target, output in result["targets"].items():
    output_rows.append({
        "target": target,
        "unit": output["unit"],
        "alpha": output["selected_alpha"],
        "confidence": output["confidence_level"],
        "baseline": output["baseline_prediction"],
        "scenario": output["scenario_prediction"],
        "incremental": output["incremental_scenario_effect"],
        "best_naive": output["naive_benchmark_comparison"]["best_mae_strategy"],
        "ridge_beat_naive_both": output["naive_benchmark_comparison"]["ridge_beat_best_naive_on_both_errors"],
    })
outputs = pd.DataFrame(output_rows)
display(outputs.round(4))
assert len(outputs) == 12
assert outputs[["baseline", "scenario", "incremental"]].notna().all().all()
assert tuple(outputs["target"]) == ALL_RIDGE_TARGETS
assert len(engine.pipelines) == 12
assert summary["naive_benchmark_substitution_count"] == 0


## Neutral behavior and validation failures

A neutral scenario produces exactly zero incremental effect for all 12 Ridge targets. Missing, extra, nonnumeric, and nonfinite inputs are rejected. Out-of-training-range shocks are never clipped: the default policy returns a warning, while the optional strict policy raises an error.

In [ ]:
neutral = predict_scenario(NEUTRAL_SCENARIO, engine=engine)
neutral_check = {
    target: neutral["targets"][target]["incremental_scenario_effect"]
    for target in ALL_RIDGE_TARGETS
}
display(pd.Series(neutral_check, name="neutral_incremental_effect").to_frame())
assert all(value == 0.0 for value in neutral_check.values())

outside = dict(EXAMPLE_SCENARIO)
outside["fed_funds_rate_change_bps"] = 76.0
warning_result = predict_scenario(outside, engine=engine)
display(pd.DataFrame(warning_result["input_range_warnings"]).loc[
    lambda frame: frame["warning_type"].eq("SCENARIO_SHOCK_OUTSIDE_TRAINING_RANGE")
])
strict_error = None
try:
    predict_scenario(outside, engine=engine, out_of_range_policy="error")
except ScenarioValidationError as exc:
    strict_error = str(exc)
assert strict_error is not None
strict_error


## Step 14 decision

The corrected all-target Ridge workflow is complete as a callable research interface and can be handed to a separate all-target XGBoost comparison if the team chooses. It is still not a production API or a production-approved model set, and Steps 15–17 have not begun.

In [ ]:
checks = pd.Series(summary["validations"], name="passed")
display(checks.to_frame())
display(artifacts["model_gate_audit"])
assert checks.all()
assert summary["locked_final_test"]["target_rows_parsed"] == 0
assert summary["locked_final_test"]["access_count"] == 0
assert summary["completion_gate"] == "STEP_14_ALL_TARGET_RIDGE_INFERENCE_COMPLETE"
assert summary["ridge_pipeline_count"] == 12
assert len(LOW_CONFIDENCE_PROMOTED_TARGETS) == 3
assert len(WEAK_EVIDENCE_RIDGE_TARGETS) == 9
print("All 12 targets: numeric target-specific Ridge research inference.")
print("Three low-confidence targets; nine weak-evidence targets with explicit warnings.")
print(summary["completion_gate"])
